In [1]:
import pysftp
import sys
import os
import pandas as pd

In [2]:
#download files again (yes/no)?
download = "no"

#set year for data creation
year = '2017'

In [3]:
#ENTSO-E sftp settings
host = "sftp-transparency.entsoe.eu"
password = "qhfWzbuxRkmmKb+"                
username = "jonas.savelsberg@unibas.ch"                
port = '22'

In [4]:
dir_out = "../parsed_data/"

In [5]:
map_country = {
'AT' : 'AT',
'RS' : 'RS',
'SK' : 'SK',
'HU' : 'HU',
'RO' : 'RO',
'ES' : 'ES',
'FR' : 'FR',
'DE_AT_LU' : 'DE',
'IT_NORD_AT' : 'IT',
'IT_NORD_FR' : 'IT',
'IT_NORD_SI' : 'IT',
'SI' : 'SI',
'NL' : 'NL',
'BE' : 'BE',
'NO5' : 'NO',
'NO3' : 'NO',
'SE2' : 'SE',
'LV' : 'LV',
'NO4' : 'NO',
'SE4' : 'SE',
'GB' : 'GB',
'PT' : 'PT',
'GR' : 'GR',
'IT_SICI' : 'IT',
'IT_BRNN' : 'IT',
'EE' : 'EE',
'NO2' : 'NO',
'SE1' : 'SE',
'NO1' : 'NO',
'DK2' : 'DK',
'FI' : 'FI',
'DK1' : 'DK',
'LT' : 'LT',
'IT_SUD' : 'IT',
'IT_North' : 'IT',
'IT_CSUD' : 'IT',
'IT_SARD' : 'IT',
'IT_CNOR' : 'IT',
'IT_ROSN' : 'IT',
'IT_NORD_CH' : 'IT',
'SE3' : 'SE',
'IT_PRGP' : 'IT',
'IT_SACO_AC' : 'IT',
'PL' : 'PL',
'IT_SACO_DC' : 'IT',
'IT_FOGN' : 'IT',
'IT_GR' : 'IT',
'BG' : 'BG',
'HR' : 'HR',
'CH' : 'CH',
'CZ' : 'CZ',
'IE_SEM' : 'IE'
              }

In [6]:
bz_countries = {'IT','SE','DK','NO'}

In [7]:
#Connect to ENTSO-E Transparency FTP
#for this to work, pysftp 0.2.8 is needed!
path = '/TP_export/'
path_local = os.path.join(os.pardir,'source_data/ENTSOE/') 

In [8]:
if download =="yes":
    with pysftp.Connection(host=host, username=username, password=password) as sftp:
        print("Connection succesfully established.")
        
        # show list of files
        files = sftp.listdir('/TP_export/')
        #print(files)

## load data

In [9]:
#set paths and get file names
path_prices = path+'DayAheadPrices/'
path_prices_local = path_local+'prices/'
if download =="yes":
    with pysftp.Connection(host=host, username=username, password=password) as sftp:
        print("Connection succesfully established.")
        # show list of files
        files = sftp.listdir(path_prices)
if download =="no": 
    files = os.listdir(path_prices_local)
if year != "":
    files = [i for i in files if year in i]

In [10]:
#download aggregated data
if download =="yes":
    with pysftp.Connection(host=host, username=username, password=password) as sftp:
        for file in files:
            sftp.get(path_prices+file,path_prices_local+file)
            print('Successfully downloaded file '+file)

In [12]:
#combine files to one data frame
df_prices = pd.DataFrame()
for file in files:
    df_temp = pd.read_csv(path_prices_local+file,
                          decimal=".",encoding="UTF-16LE",sep="\t",
                          parse_dates=True, index_col="DateTime")
    df_prices = df_prices.append(df_temp)
df_prices = df_prices.drop(['Year','Month','Day','ResolutionCode','AreaCode','AreaTypeCode','AreaName','UpdateTime'], axis=1)
df_prices.head()

""


Convert to EUR using exchange rates from https://www.ecb.europa.eu/stats/policy_and_exchange_rates/euro_reference_exchange_rates/html/eurofxref-graph-bgn.en.html
- GBP-EUR: 1.1413
- RON-EUR: 0.2189
- PLN-EUR: 0.2349
- BGN-EUR: 0.5113      

In [ ]:
df_prices.Price[df_prices.Currency == 'GBP'] = df_prices.Price * 1.1413
df_prices.Price[df_prices.Currency == 'RON'] = df_prices.Price * 0.2189
df_prices.Price[df_prices.Currency == 'PLN'] = df_prices.Price * 0.2349
df_prices.Price[df_prices.Currency == 'BGN'] = df_prices.Price * 0.5113
df_prices = df_prices.drop(columns = 'Currency').reset_index().rename(columns={'DateTime':'date'}).set_index(['date','MapCode'])
df_prices.head()

In [ ]:
df_prices_pivot = df_prices.reset_index().pivot_table(index='date',columns='MapCode',values='Price')
df_prices_hourly = df_prices_pivot.resample('H').mean()
df_prices_hourly.head()

In [ ]:
df_prices_stack = pd.DataFrame(df_prices_hourly.stack()).rename(columns={0:'EUR_per_MWh'}).reset_index()
df_prices_stack['country'] = df_prices_stack.MapCode.map(map_country)
df_prices_stack.head()

now we also load load values for weighting in countries with multiple bidding zones

In [ ]:
#set paths and get file names
path_load = path+'ActualTotalLoad/'
path_load_local = path_local+'load/'
if download =="yes":
    with pysftp.Connection(host=host, username=username, password=password) as sftp:
        print("Connection succesfully established.")
        # show list of files
        files = sftp.listdir(path_load)
if download =="no": 
    files = os.listdir(path_load_local)
if year != "":
    files = [i for i in files if year in i]

In [ ]:
#download aggregated load data (ActualTotalLoad)
if download =="yes":
    with pysftp.Connection(host=host, username=username, password=password) as sftp:
        for file in files:
            sftp.get(path_load+file,path_load_local+file)
            print('Successfully downloaded file '+file)

In [ ]:
#combine files to one data frame
df_load = pd.DataFrame()
for file in files:
    df_temp = pd.read_csv(path_load_local+file,
                          decimal=".",encoding="UTF-16LE",sep="\t",
                          parse_dates=True, index_col="DateTime").drop("areacode", axis=1)
    df_load = df_load.append(df_temp)
df_load = df_load[df_load.AreaTypeCode == "BZN"].drop(["AreaTypeCode",'Year','Month','Day','ResolutionCode','AreaName','UpdateTime'], axis=1).reset_index()
df_load = df_load.rename(columns={'DateTime':'date'})
df_load = df_load
df_load.head()

In [ ]:
#resample to hourly
df_load_pivot = df_load.pivot_table(index="date",columns="MapCode",values='TotalLoadValue')
df_load_pivot_hourly = df_load_pivot.resample('1H').mean()
df_load_hourly = pd.DataFrame(df_load_pivot_hourly.stack()).rename(columns={0:'TotalLoadValue'}).reset_index()
df_load_hourly['country'] = df_load_hourly.MapCode.map(map_country)
df_load_hourly = df_load_hourly[df_load_hourly.country.isin(bz_countries)]
df_load_hourly.head()

In [ ]:
df_load_country = df_load_hourly.groupby(['date','country']).sum().reset_index()
df_load_country.head()

In [ ]:
df_load_hourly = df_load_hourly.drop(columns='country')

In [ ]:
df_prices_temp = df_prices_stack.merge(df_load_hourly,on=['date','MapCode'],how='left')
df_prices_temp_2 = df_prices_temp.merge(df_load_country,on=['date','country'],how='left')
df_prices_temp_2['weight'] = df_prices_temp_2.TotalLoadValue_x / df_prices_temp_2.TotalLoadValue_y
df_prices_temp_2['weight'][~df_prices_temp_2.country.isin(bz_countries)] = df_prices_temp_2['weight'].fillna(1)
df_prices_temp_2['EUR_per_MWh'] = df_prices_temp_2['EUR_per_MWh'] * df_prices_temp_2['weight']
df_prices_temp_2.tail()

In [ ]:
df_prices_hourly_stack = df_prices_temp_2.groupby(['date','country']).sum().reset_index()
df_prices_hourly_stack.head()

In [ ]:
df_prices_hourly = df_prices_hourly_stack[['date','country','EUR_per_MWh']].pivot_table(index='date',columns='country',values='EUR_per_MWh')
df_prices_hourly.head()

In [ ]:
df_prices_hourly['LU']=df_prices_hourly['DE']

In [ ]:
df_prices_hourly.to_csv(dir_out+'prices_'+year+'_hourly_entsoe.csv', encoding="utf-8")